In [1]:
import duckdb
con = duckdb.connect()
Y = "'../data/raw/yellow_tripdata_2026-08.parquet'"

con.sql(f"DESCRIBE SELECT * FROM {Y}").df()

,column_name,column_type,null,key,default,extra
0,VendorID,INTEGER,YES,None,None,None
1,tpep_pickup_datetime,TIMESTAMP,YES,None,None,None
2,tpep_dropoff_datetime,TIMESTAMP,YES,None,None,None
3,passenger_count,BIGINT,YES,None,None,None
4,trip_distance,DOUBLE,YES,None,None,None
5,RatecodeID,BIGINT,YES,None,None,None
6,store_and_fwd_flag,VARCHAR,YES,None,None,None
7,PULocationID,INTEGER,YES,None,None,None
8,DOLocationID,INTEGER,YES,None,None,None
9,payment_type,BIGINT,YES,None,None,None


In [2]:
con.sql(f"SUMMARIZE SELECT * FROM {Y}").df()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,column_name,column_type,min,max,approx_unique,avg,std,q25,q50,q75,count,null_percentage
0,VendorID,INTEGER,1,7,4,1.8956557885058243,0.7072127929883647,2,2,2,3336716,0.00
1,tpep_pickup_datetime,TIMESTAMP,2009-01-01 14:39:49,2026-08-31 23:59:59,1695632,2026-08-16 05:32:10.135508,NaN,2026-08-08 06:48:42.121568,2026-08-15 21:02:08.147729,2026-08-23 20:23:32.103416,3336716,0.00
2,tpep_dropoff_datetime,TIMESTAMP,2009-01-01 15:03:32,2026-09-01 20:16:00,1774306,2026-08-16 05:49:11.696285,NaN,2026-08-08 06:48:56.181188,2026-08-15 22:40:59.583957,2026-08-23 19:40:40.126794,3336716,0.00
3,passenger_count,BIGINT,0,8,8,1.2689967232931836,0.6724422490918067,1,1,1,3336716,27.61
4,trip_distance,DOUBLE,0.0,223198.74,5094,4.9042181084627945,420.5507841866997,1.063942323661179,1.9859616300514946,4.215739140959595,3336716,0.00
5,RatecodeID,BIGINT,1,99,6,4.228766298149271,17.200366853495247,1,1,1,3336716,27.61
6,store_and_fwd_flag,VARCHAR,N,Y,2,NaN,NaN,NaN,NaN,NaN,3336716,27.61
7,PULocationID,INTEGER,1,265,261,160.36378702892304,66.56630558848146,115,161,232,3336716,0.00
8,DOLocationID,INTEGER,1,265,261,159.3404221396127,70.8187918071399,107,161,232,3336716,0.00
9,payment_type,BIGINT,0,4,5,0.850080438371141,0.6544802984562224,0,1,1,3336716,0.00


In [ ]:
con.sql(f"""
SELECT
  COUNT(*) AS total_rows,
  COUNT(*) FILTER (WHERE tpep_pickup_datetime <  '2026-08-01'
                      OR tpep_pickup_datetime >= '2026-09-01')            AS outside_month,
  COUNT(*) FILTER (WHERE fare_amount < 0)                                 AS negative_fare,
  COUNT(*) FILTER (WHERE trip_distance = 0)                               AS zero_distance,
  COUNT(*) FILTER (WHERE tpep_dropoff_datetime <= tpep_pickup_datetime)   AS dropoff_before_pickup,
  COUNT(*) FILTER (WHERE passenger_count IS NULL)                         AS null_passengers,
  COUNT(*) FILTER (WHERE datediff('hour', tpep_pickup_datetime,
                                  tpep_dropoff_datetime) > 24)            AS over_24h
FROM {Y}
""").df()

,total_rows,outside_month,negative_fare,zero_distance,negative_distance,dropoff_before_pickup,null_passengers,over_24h
0,3336716,15,14049,119059,0,43185,921181,42
